# Build a miniscope recording, one stage at a time

A 1-photon miniscope recording is the end of a physical chain: neurons fire →
calcium binds an indicator → light is emitted → optics blur and dim it → the
brain moves → a sensor counts noisy photons. The minian *analysis* pipeline runs
that chain **backwards** to recover the neurons. Here we run it **forwards** —
and we build it **together**, choosing the physics as we go.

Each stage has the same rhythm:

1. **Understand** — what this piece of physics is, and what to watch.
2. **Explore** — drag sliders and *see* the effect.
3. **Commit** — set the values you want, add the stage, and move on.

The recording grows as we add stages; by the end we have a full synthetic movie
*and* the exact ground truth behind it — positions, traces, footprints, motion,
and which cells are even physically recoverable. That truth is what Notebook 2
scores the real pipeline against.

> The interactive sliders need a **live kernel** — run this notebook (don't just
> read it). Run cells top to bottom; each stage uses the choices committed above
> it.

## Setup

The simulator is `minian.simulation`. We also pull in `mediapy` (smooth inline video playback) and a couple of tiny plotting helpers. `mediapy` finds `ffmpeg` on your PATH automatically.

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter
from ipywidgets import interact, FloatSlider, IntSlider
import mediapy

from minian.simulation import (
    Acquisition, Optics, ImageSensor, Tissue, Output, Spec, simulate,
    PlaceSomata, CellActivity, CellOptics, Render,
    Neuropil, Bleaching, BrainMotion, Vignette, Leakage, Sensor,
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 3.2)
plt.rcParams["image.cmap"] = "magma"


def play(movie, fps=20, height=280, title=None):
    # normalize a (frame, h, w) movie to [0,1] and show a looping inline video
    arr = np.asarray(movie, dtype=float)
    lo, hi = float(arr.min()), float(arr.max())
    mediapy.show_video((arr - lo) / (hi - lo + 1e-9), fps=fps, height=height, title=title, codec="h264")


def montage(footprints, titles=None, n=6, cmap="magma"):
    # show the first n 2-D footprints side by side
    n = min(n, len(footprints))
    fig, axes = plt.subplots(1, n, figsize=(2.1 * n, 2.3))
    axes = np.atleast_1d(axes)
    for k, ax in enumerate(axes):
        ax.imshow(footprints[k], cmap=cmap); ax.axis("off")
        if titles is not None:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout(); plt.show()

## Our recording, as we build it

Two things carry forward as we go:

- **`acq`** — the *scope*: optics, sensor, tissue, frame rate, duration. We set
  this first (Stage 1).
- **`steps`** — the ordered list of construction stages. We start empty and
  **append one stage per section** as we commit to it.

`build()` simulates the recording from whatever stages we've committed so far;
`preview()` does the same on a small, fast FOV for responsive sliders. (`preview`
keeps the real pixel size and optics — so blur and brightness look *exactly*
right — and just images a smaller patch of tissue for a few seconds.)

In [18]:
SEED = 7
steps = []          # we append one StepSpec per stage, as we commit to it

# acq is created in Stage 1; these helpers read whatever it currently is.
def build(until=None):
    # full-resolution recording from the stages committed so far
    spec = Spec(acquisition=acq, seed=SEED, output=Output(save_intermediates=True), steps=list(steps))
    return simulate(spec, until=until)

def preview(extra=None, until=None, n_px=128, duration_s=3.0):
    # fast, small-FOV recording for slider exploration: same pixel size + optics,
    # just a smaller patch and fewer frames. `extra` is a step to try on top of
    # the committed ones without appending it.
    fast = acq.model_copy(update={
        "image_sensor": acq.image_sensor.model_copy(update={"n_px_height": n_px, "n_px_width": n_px}),
        "duration_s": duration_s,
    })
    trial = list(steps) + ([extra] if extra is not None else [])
    spec = Spec(acquisition=fast, seed=SEED, output=Output(save_intermediates=True), steps=trial)
    return simulate(spec, until=until)

## Stage 1 — The scope (optics + sensor)

Everything downstream lives inside the scope you choose here. Two relationships
do most of the work:

- **Field of view & zoom.** The sensor chip is fixed hardware. **Magnification**
  sets the field of view, `FOV = sensor size / magnification` — higher mag zooms
  in, so the cell fills more of the frame. **Pixel pitch** sets how many pixels
  tile that same chip (the FOV is unchanged): finer pitch → more, smaller pixels.
  Pixel size in tissue is `pitch / magnification`.
- **Resolution floor.** Diffraction sets the sharpest a point can ever be,
  `σ ≈ 0.21·λ/NA`. A 1-photon miniscope has a *low* NA, so this floor is real.
- **Light collection.** The objective only gathers light over a cone whose solid
  angle scales as **NA²**, so a low-NA scope is *fundamentally dimmer* — not just
  blurrier. Both effects push the same way: higher NA is sharper *and* brighter.

**Explore:** what you see is **what the image sensor sees** — its pixel grid,
imaging a single cell. **Magnification** zooms the field of view in and out, so
the cell grows and shrinks in the frame. **Pixel pitch** re-samples the same chip
(the pixel count changes, the FOV does not — watch the footprint go blocky vs
fine). **NA** sharpens the cell and, on the log-scaled fixed-max colorbar,
brightens it (∝ NA²).

In [19]:
CHIP_UM = 120.0   # fixed sensor chip extent: n_px = CHIP/pitch, object FOV = CHIP/magnification
_SOMA_UM = 6.0    # soma radius (um)
# no-photon pixels (0 -> log undefined / below vmin) render black, not transparent-white
_CMAP = plt.cm.magma.copy(); _CMAP.set_bad("black"); _CMAP.set_under("black")

def _image_scene(na, magnification, pixel_pitch_um):
    # what the sensor sees: its n_px x n_px grid imaging one centered cell.
    n_px = int(np.clip(round(CHIP_UM / pixel_pitch_um), 8, 256))   # pitch -> pixel count
    px_um = pixel_pitch_um / magnification                         # object-space pixel size
    fov_um = n_px * px_um                                          # == CHIP / magnification
    soma_px = _SOMA_UM / px_um                                     # cell radius, in sensor pixels
    c = (n_px - 1) / 2
    yy, xx = np.mgrid[0:n_px, 0:n_px]
    img = (np.hypot(yy - c, xx - c) <= soma_px).astype(float)
    optics = Optics(na=na, magnification=magnification, emission_nm=525.0)
    img = gaussian_filter(img, optics.diffraction_sigma_um / px_um) * optics.collection_efficiency
    return img, px_um, n_px, fov_um, optics

def explore_scope(na=0.18, magnification=3.0, pixel_pitch_um=4.8):
    img, px_um, n_px, fov_um, optics = _image_scene(na, magnification, pixel_pitch_um)
    vmax = _image_scene(0.8, magnification, pixel_pitch_um)[0].max()   # brightest NA -> fixed scale
    fig, ax = plt.subplots(figsize=(4.8, 4.0))
    im = ax.imshow(img, norm=LogNorm(vmin=vmax / 1e3, vmax=vmax), cmap=_CMAP, interpolation="nearest")
    ax.set_title(f"sensor {n_px}x{n_px} px  |  mag {magnification:g}x -> FOV {fov_um:.0f} um  |  "
                 f"{px_um:.2f} um/px\n"
                 f"diffraction sigma {optics.diffraction_sigma_um*1000:.0f} nm  |  "
                 f"brightness (NA^2) = {optics.collection_efficiency:.3f}", fontsize=9)
    ax.set(xlabel="sensor pixel", ylabel="sensor pixel")
    fig.colorbar(im, ax=ax, label="intensity (log)")
    plt.tight_layout(); plt.show()

interact(explore_scope,
         na=FloatSlider(min=0.1, max=0.8, step=0.02, value=0.18, continuous_update=False),
         magnification=FloatSlider(min=1.0, max=10.0, step=0.5, value=3.0, continuous_update=False),
         pixel_pitch_um=FloatSlider(min=1.0, max=8.0, step=0.2, value=4.8, continuous_update=False));

interactive(children=(FloatSlider(value=0.18, continuous_update=False, description='na', max=0.8, min=0.1, ste…

**Commit the scope.** These are the values we carry forward (a low-NA, wide-field deep-imaging miniscope). The focal plane sits at 200 µm — we'll place cells around it next.

In [21]:
acq = Acquisition(
    fps=20.0,
    duration_s=30.0,
    optics=Optics(na=0.18, magnification=3.0, emission_nm=525.0,
                  focal_plane_um=200.0, depth_of_field_um=15.0),
    image_sensor=ImageSensor(n_px_height=300, n_px_width=300, pixel_pitch_um=4.8, bit_depth=8),
    tissue=Tissue(scatter_mfp_um=100.0, scatter_blur_per_um=0.02),
)
print(f"FOV {acq.fov_um[0]:.0f} x {acq.fov_um[1]:.0f} um | {acq.pixel_size_um:.2f} um/px "
      f"| {acq.n_frames} frames @ {acq.fps:.0f} fps | focal plane {acq.optics.focal_plane_um:.0f} um")

FOV 480 x 480 um | 1.60 um/px | 600 frames @ 20 fps | focal plane 200 um


## Stage 2 — Place the somata

Now we put cell bodies into the tissue volume — lateral position plus a **depth**
`z`. Depth is the quiet protagonist of this whole notebook: it decides how blurred
and how dim each cell ends up, and ultimately whether it is recoverable at all.

**Explore:** drag the density, the depth band (relative to the 200 µm focal
plane), and the soma size. The left panel is where cells sit (color = depth); the
right shows a few of their sharp, optics-free footprints (`A_planted`) — the ideal
target the analysis would love to recover.

In [22]:
def explore_placement(density_per_mm2=700.0, depth_lo=180.0, depth_hi=220.0, soma_radius_um=5.0):
    rec = preview(extra=PlaceSomata(density_per_mm2=density_per_mm2, soma_radius_um=soma_radius_um,
                                    depth_range_um=(depth_lo, depth_hi)),
                  until="place_somata")
    g = rec.ground_truth
    z, y, x = g.centers_um[:, 0], g.centers_um[:, 1], g.centers_um[:, 2]
    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    sc = ax.scatter(x, y, c=z, s=30, cmap="viridis"); ax.invert_yaxis()
    ax.set(title=f"{g.n_units} cells in this preview FOV", xlabel="x (um)", ylabel="y (um)")
    plt.colorbar(sc, ax=ax, label="depth z (um)"); plt.tight_layout(); plt.show()
    if g.n_units:
        montage(g.A_planted, titles=[f"z={zz:.0f}" for zz in z], n=6)

interact(explore_placement,
         density_per_mm2=FloatSlider(min=100, max=2000, step=100, value=700, continuous_update=False),
         depth_lo=FloatSlider(min=0, max=250, step=10, value=180, continuous_update=False),
         depth_hi=FloatSlider(min=0, max=300, step=10, value=220, continuous_update=False),
         soma_radius_um=FloatSlider(min=2, max=12, step=0.5, value=5, continuous_update=False));

interactive(children=(FloatSlider(value=700.0, continuous_update=False, description='density_per_mm2', max=200…

**Commit the somata.** Append the stage with our chosen values.

In [23]:
steps.append(PlaceSomata(density_per_mm2=700.0, soma_radius_um=5.0, depth_range_um=(180.0, 220.0)))

rec = build(until="place_somata"); gt = rec.ground_truth
z = gt.centers_um[:, 0]
print(f"placed {gt.n_units} cells | depth {z.min():.0f}-{z.max():.0f} um "
      f"| focal plane at {acq.optics.focal_plane_um:.0f} um")
print("(which cells are in focus / detectable is decided at the optics stage, next)")

placed 161 cells | depth 180-220 um | focal plane at 200 um
(which cells are in focus / detectable is decided at the optics stage, next)


## Stage 3 — Calcium activity

Each cell now gets a spike train and the fluorescence it produces. Spikes come
from a 2-state (quiescent/active) gate driving a Poisson process; each spike adds
a calcium transient with a fast rise and slow decay,
`k(t) = e^{-t/τ_d} − e^{-t/τ_r}`. This rise/decay shape is exactly what
deconvolution later assumes.

**Explore:** the active firing rate and the decay time `τ_d`. Watch the traces:
faster rate → busier; longer `τ_d` → fatter, slower-falling transients. (The
sampling rule `τ_d · fps ≳ 1` must hold or the decay is unresolvable.)

In [24]:
def explore_activity(active_rate_hz=2.0, tau_decay_s=0.5):
    rec = preview(extra=CellActivity(active_rate_hz=active_rate_hz, tau_decay_s=tau_decay_s),
                  until="cell_activity", duration_s=20.0)
    g = rec.ground_truth
    t = np.arange(rec.spec.acquisition.n_frames) / rec.spec.acquisition.fps
    busiest = np.argsort(g.S.sum(axis=1))[-4:] if g.n_units else []
    fig, ax = plt.subplots(figsize=(9, 3.2))
    for k, u in enumerate(busiest):
        ax.plot(t, g.C[u] + k * 1.2, lw=1.1)
        spk = np.where(g.S[u] > 0)[0]
        ax.plot(t[spk], np.full_like(spk, k * 1.2, dtype=float) - 0.2, "|", color="k", ms=6)
    ax.set(title=f"traces C (lines) + spikes S (ticks) | rate {active_rate_hz} Hz, tau_d {tau_decay_s}s",
           xlabel="time (s)", ylabel="cell (offset)")
    plt.tight_layout(); plt.show()

interact(explore_activity,
         active_rate_hz=FloatSlider(min=0.5, max=8.0, step=0.5, value=2.0, continuous_update=False),
         tau_decay_s=FloatSlider(min=0.1, max=1.5, step=0.1, value=0.5, continuous_update=False));

interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='active_rate_hz', max=8.0, m…

**Commit the activity.** Append the stage; the cells now have traces.

In [25]:
steps.append(CellActivity(active_rate_hz=2.0, tau_decay_s=0.5))
print("stages committed so far:", " -> ".join(s.kind for s in steps))

stages committed so far: place_somata -> cell_activity


---

### Next: the cells become a movie

So far we have cells with positions, depths, and traces — but no image yet. The
next stages turn them into a movie and then degrade it the way the physical world
does: **optics** (blur + dim by depth), **render** (the first actual video!),
then **background, bleaching, motion, illumination, and sensor noise**. Those
stages produce real video at full resolution, so we'll decide *there* how to
balance slider responsiveness against fidelity — stage by stage.

*(More stages will be added below.)*